*Módulo 3 de 9*

> **Prefer English?** Open [`03_clean_data_geomedian.ipynb`](../en/03_clean_data_geomedian.ipynb) — it is the same module, in English.


# ☁️ Módulo 3 — Datos limpios: de las nubes a la geomediana

🧭 **Objetivos** — entender por qué las imágenes satelitales crudas son un
desorden (nubes, sombras, huecos), qué hacen una **máscara de nubes** y las
**banderas de calidad**, y cómo una **geomediana** convierte todo un mes de
imágenes imperfectas en una sola imagen limpia y sin huecos — el tile que
has estado usando.

📚 **El problema.** Un solo paso del satélite suele arruinarse por nubes y
sus sombras. Una imagen de tu parcela podría ser 40% nube. La solución es la
**composición temporal**: tomar *muchas* imágenes en un periodo y
combinarlas por píxel, quedándose solo con las observaciones buenas.

📚 **La geomediana.** Una mediana simple por banda tomaría, digamos, el rojo
mediano de una fecha y el NIR mediano de otra — rompiendo el color real del
píxel. La **geomediana** (mediana geométrica) en cambio encuentra el único
valor multibanda más cercano a todas las observaciones sin nube *a la vez*,
así los cocientes como el NDVI se mantienen físicamente consistentes. Las
nubes son atípicas, así que quedan descartadas. El resultado son **Datos
Listos para Análisis**: reflectancia de superficie, sin nubes, lista.

![la geomediana](../../anim/es/03_geomedian.svg)


## De dónde salen los datos (y el GEE opcional)

El tile se construye con **NASA HLS** (Landsat + Sentinel-2 armonizados),
transmitido desde catálogos abiertos **STAC/COG** — sin cuenta. El pipeline
de producción (`geocrop_analysis_mx`) puede *opcionalmente* usar también
Google Earth Engine, pero no es obligatorio: la misma geomediana se puede
construir desde catálogos abiertos. Verás esta elección de nuevo en el
Módulo 9.


## Simula el problema, luego la solución

No tienes la pila cruda con nubes en el navegador, pero puedes *sentir* por
qué funciona la geomediana con un experimento diminuto: toma un valor limpio
de píxel, salpica encima observaciones "nubladas" falsas (las nubes son
brillantes — valores altos), y observa cómo la **mediana** las ignora
mientras la **media** se deja engañar.


In [ ]:
import numpy as np

# 10 observaciones de la reflectancia roja de un píxel durante un mes.
# 7 están despejadas (~0.06); 3 tienen nube (brillantes, ~0.8).
observaciones = np.array([0.06, 0.05, 0.80, 0.07, 0.06, 0.78, 0.05, 0.82, 0.06, 0.07])

print("Media    (engañada por nubes):", round(observaciones.mean(), 3))
print("Mediana  (descarta las nubes):", round(np.median(observaciones), 3))
print("El valor limpio real es ~0.06 — la mediana lo recupera.")

## La geomediana ya está en tu tile

Cada píxel de tu tile es el *resultado* de este proceso aplicado a lo largo
de un mes de imágenes HLS, para todas las bandas juntas. Por eso se ve sin
costuras — sin huecos de nube, color consistente. Cárgalo y confirma que
está completo (sin píxeles faltantes en el área válida).


In [ ]:
# Trae el tile del taller (pocos MB; queda en caché tras la primera descarga)
import os, sys

async def trae_archivo(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

TILE = await trae_archivo("crop_tile_384.tif")
print("Tile listo:", TILE)

In [ ]:
import rasterio, matplotlib.pyplot as plt

with rasterio.open(TILE) as src:
    img = src.read()

# Una geomediana no tiene huecos de nube: cuenta píxeles exactamente 0 (nodata)
validos = np.count_nonzero(img[3] != 0)       # banda NIR
total = img[3].size
print(f"Píxeles válidos: {validos:,} de {total:,} ({100*validos/total:.1f}%)")

rgb = np.clip(np.dstack([img[2], img[1], img[0]]) / 3000.0, 0, 1)
plt.figure(figsize=(7, 7)); plt.imshow(rgb)
plt.title("Geomediana limpia — sin nubes, sin huecos"); plt.axis("off"); plt.show()

## 🧪 Ponte a prueba

**¿Por qué no simplemente promediar todas las imágenes de un mes para quitar
las nubes?**

<details><summary>Ver respuesta</summary>

Las nubes son atípicos brillantes; la **media** se jala hacia arriba por
ellas. La **mediana** (y la geomediana multibanda) ignora los atípicos, así
que las observaciones con nube quedan descartadas. Promediar dejaría una
imagen brumosa y teñida de nube.

</details>

**¿Qué te da el "geo" de geomediana frente a una mediana por banda?**

<details><summary>Ver respuesta</summary>

Mantiene las bandas *consistentes por píxel*: en vez de mezclar el rojo de
una fecha con el NIR de otra, elige una observación multibanda cercana a
todas las despejadas a la vez. Eso mantiene los cocientes como el NDVI
físicamente con sentido.

</details>


## 🔭 Profundiza

Opcional: estas tarjetas bilingües de conceptos amplían lo que acabas
de aprender (prerrequisitos, linaje a fundamentos, referencias):

- [Enmascarado de nubes](https://abxda.github.io/rs-learning-audio/?id=cloud-masking&lang=es)
- [Banderas de calidad](https://abxda.github.io/rs-learning-audio/?id=quality-flags&lang=es)
- [Datos Listos para Análisis (ARD)](https://abxda.github.io/rs-learning-audio/?id=analysis-ready-data&lang=es)
- [Composición temporal](https://abxda.github.io/rs-learning-audio/?id=temporal-compositing&lang=es)
- [Landsat-Sentinel Armonizado (HLS)](https://abxda.github.io/rs-learning-audio/?id=harmonized-landsat-sentinel&lang=es)
- [Reflectancia de superficie](https://abxda.github.io/rs-learning-audio/?id=surface-reflectance&lang=es)



---

[← Anterior · Módulo 2 — Cómo ve el mundo un satélite](02_como_ve_un_satelite.ipynb) · [Siguiente → · Módulo 4 — Índices de vegetación](04_indices_de_vegetacion.ipynb)
